# 04 — GAN 3D Normalized
**Dự án:** Latent Manipulation of Brain MRI using Volume-Preserving GANs

**Input:** NIfTI normalized từ `00b_preprocessing_3d.ipynb`

**Kiến trúc:**
- **Generator**: 3D U-Net + Age Embedding inject vào bottleneck
- **Discriminator**: 3D PatchGAN + Age Conditioning

**Output:**
```
gan3d_normalized/
└── best_model.pth
```

## Bước 1: Cấu hình

In [1]:
import os

# ==== ĐƯỜNG DẪN ====
DATA_DIR   = '/kaggle/input/datasets/minhbodoi/dghgyfgfh/normalized'
LABELS_CSV = '/kaggle/input/datasets/minhbodoi/dghgyfgfh/preprocessing_log.csv'
OUTPUT_DIR = '/kaggle/working/gan3d_normalized'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==== HYPERPARAMETERS ====
VOLUME_SIZE = 64      # resize về 64×64×64 vì 3D nặng hơn 2D rất nhiều
BATCH_SIZE  = 1       # 3D nặng, chỉ dùng batch=1
NUM_EPOCHS  = 100
LR_G        = 2e-4
LR_D        = 2e-4
LAMBDA_L1   = 100
LAMBDA_AGE  = 1
LATENT_DIM  = 256

# Tìm file .nii hoặc .nii.gz (Kaggle tự giải nén .nii.gz thành .nii)
nii_files = [f for f in os.listdir(DATA_DIR)
             if f.endswith('.nii.gz') or f.endswith('.nii')]
print(f'Data dir : {DATA_DIR}')
print(f'NII files: {len(nii_files)}')

Data dir : /kaggle/input/datasets/minhbodoi/dghgyfgfh/normalized
NII files: 10


## Bước 2: Import thư viện

In [2]:
!pip install nibabel -q

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU : Tesla T4
VRAM: 15.6 GB


## Bước 3: Dataset

In [3]:
def find_nii(data_dir, subject_id):
    """Tìm file .nii hoặc .nii.gz — xử lý cả 2 trường hợp."""
    for ext in ['.nii.gz', '.nii']:
        path = os.path.join(data_dir, f'{subject_id}{ext}')
        if os.path.exists(path):
            return path
    return None


class BrainMRI3DDataset(Dataset):
    def __init__(self, data_dir, labels_csv, volume_size=64):
        """
        Load file NIfTI 3D, resize về volume_size×volume_size×volume_size.
        Resize nhỏ để tiết kiệm RAM khi train 3D.
        """
        self.data_dir    = data_dir
        self.volume_size = volume_size

        df = pd.read_csv(labels_csv)
        df['nii_path'] = df['subject_id'].apply(
            lambda x: find_nii(data_dir, x)
        )
        df = df[df['nii_path'].notna()].reset_index(drop=True)

        self.df      = df
        self.age_min = df['age'].min()
        self.age_max = df['age'].max()

        print(f'Dataset: {len(df)} subjects | tuổi [{self.age_min:.1f}, {self.age_max:.1f}]')

    def normalize_age(self, age):
        return 2 * (age - self.age_min) / (self.age_max - self.age_min) - 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        data = nib.load(row['nii_path']).get_fdata().astype(np.float32)

        # Resize về volume_size
        vol = torch.tensor(data).unsqueeze(0).unsqueeze(0)  # (1,1,H,W,D)
        vol = F.interpolate(vol, size=(self.volume_size,) * 3,
                            mode='trilinear', align_corners=False)
        vol = vol.squeeze(0)  # (1, V, V, V)

        # Normalize về [-1, 1]
        vol = vol * 2 - 1

        age_norm = torch.tensor(self.normalize_age(row['age']), dtype=torch.float32)
        age_raw  = torch.tensor(row['age'] / 100.0,            dtype=torch.float32)
        return vol, age_norm, age_raw


dataset    = BrainMRI3DDataset(DATA_DIR, LABELS_CSV, VOLUME_SIZE)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)

Dataset: 10 subjects | tuổi [20.2, 55.4]


## Bước 4: Kiến trúc Model 3D

Giống file 01 nhưng thay `Conv2d` → `Conv3d`, `ConvTranspose2d` → `ConvTranspose3d`.

In [4]:
class AgeEmbedding(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128), nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
    def forward(self, age):
        return self.net(age.unsqueeze(-1))


class UNetBlock3D(nn.Module):
    """Block 3D: Conv3d thay vì Conv2d."""
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False):
        super().__init__()
        layers = []
        if down : layers.append(nn.Conv3d(in_ch, out_ch, 4, 2, 1, bias=False))
        else    : layers.append(nn.ConvTranspose3d(in_ch, out_ch, 4, 2, 1, bias=False))
        if use_bn  : layers.append(nn.BatchNorm3d(out_ch))
        if dropout : layers.append(nn.Dropout(0.5))
        layers.append(nn.LeakyReLU(0.2) if down else nn.ReLU())
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)


class Generator3D(nn.Module):
    """
    3D U-Net Generator với age conditioning.
    Input : volume (B, 1, 64, 64, 64) + tuổi (B,)
    Output: volume sinh ra (B, 1, 64, 64, 64)
    """
    def __init__(self, latent_dim=256):
        super().__init__()
        self.age_embed = AgeEmbedding(latent_dim)
        self.age_proj  = nn.Linear(latent_dim, 256)

        # Encoder
        self.e1 = UNetBlock3D(1,   32,  down=True, use_bn=False)  # 32
        self.e2 = UNetBlock3D(32,  64,  down=True)                  # 16
        self.e3 = UNetBlock3D(64,  128, down=True)                  # 8
        self.e4 = UNetBlock3D(128, 256, down=True, use_bn=False)    # 4 bottleneck

        # Decoder
        self.d1 = UNetBlock3D(256, 128, down=False, dropout=True)  # 8
        self.d2 = UNetBlock3D(256, 64,  down=False)                 # 16
        self.d3 = UNetBlock3D(128, 32,  down=False)                 # 32

        self.out = nn.Sequential(
            nn.ConvTranspose3d(64, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x, age):
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        z  = e4 + self.age_proj(self.age_embed(age)).view(-1, 256, 1, 1, 1)
        d1 = self.d1(z)
        d2 = self.d2(torch.cat([d1, e3], dim=1))
        d3 = self.d3(torch.cat([d2, e2], dim=1))
        return self.out(torch.cat([d3, e1], dim=1))


class Discriminator3D(nn.Module):
    """
    3D PatchGAN Discriminator với age conditioning.
    Input : volume (B, 1, 64, 64, 64) + age channel → (B, 2, 64, 64, 64)
    """
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv3d(2,  32,  4, 2, 1, bias=False), nn.LeakyReLU(0.2),
            nn.Conv3d(32, 64,  4, 2, 1, bias=False), nn.BatchNorm3d(64),  nn.LeakyReLU(0.2),
            nn.Conv3d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm3d(128), nn.LeakyReLU(0.2),
            nn.Conv3d(128, 1,  4, 1, 1)
        )
    def forward(self, vol, age):
        age_map = age.view(-1, 1, 1, 1, 1).expand(
            -1, 1, vol.shape[2], vol.shape[3], vol.shape[4]
        )
        return self.model(torch.cat([vol, age_map], dim=1))


G = Generator3D(LATENT_DIM).to(DEVICE)
D = Discriminator3D().to(DEVICE)
print(f'Generator3D    : {sum(p.numel() for p in G.parameters())/1e6:.1f}M params')
print(f'Discriminator3D: {sum(p.numel() for p in D.parameters())/1e6:.1f}M params')

Generator3D    : 6.3M params
Discriminator3D: 0.7M params


## Bước 5: Loss + Optimizers

In [5]:
criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1  = nn.L1Loss()
criterion_age = nn.MSELoss()

age_regressor = nn.Sequential(
    nn.AdaptiveAvgPool3d(4),
    nn.Flatten(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEVICE)

opt_G = optim.Adam(
    list(G.parameters()) + list(age_regressor.parameters()),
    lr=LR_G, betas=(0.5, 0.999)
)
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

scheduler_G = optim.lr_scheduler.StepLR(opt_G, step_size=50, gamma=0.5)
scheduler_D = optim.lr_scheduler.StepLR(opt_D, step_size=50, gamma=0.5)

with torch.no_grad():
    d_out_shape = D(
        torch.zeros(1, 1, VOLUME_SIZE, VOLUME_SIZE, VOLUME_SIZE).to(DEVICE),
        torch.zeros(1).to(DEVICE)
    ).shape
print(f'D output shape: {d_out_shape}')
print('Loss + Optimizers sẵn sàng!')

D output shape: torch.Size([1, 1, 7, 7, 7])
Loss + Optimizers sẵn sàng!


## Bước 6: Training

In [6]:
best_loss_G = float('inf')
history     = {'loss_G': [], 'loss_D': [], 'loss_L1': [], 'loss_age': []}

print(f'Bắt đầu training {NUM_EPOCHS} epochs...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    G.train(); D.train()
    epoch_loss_G = epoch_loss_D = epoch_loss_L1 = epoch_loss_age = 0

    for real_vols, ages_norm, ages_raw in tqdm(dataloader,
                                               desc=f'Epoch {epoch}/{NUM_EPOCHS}',
                                               leave=False):
        real_vols  = real_vols.to(DEVICE)
        ages_norm  = ages_norm.to(DEVICE)
        ages_raw   = ages_raw.to(DEVICE)
        B          = real_vols.size(0)
        real_label = torch.ones(B,  *d_out_shape[1:]).to(DEVICE)
        fake_label = torch.zeros(B, *d_out_shape[1:]).to(DEVICE)

        # Train Discriminator
        opt_D.zero_grad()
        with torch.no_grad():
            fake_vols = G(real_vols, ages_norm)
        loss_D = (
            criterion_gan(D(real_vols, ages_norm), real_label) +
            criterion_gan(D(fake_vols, ages_norm), fake_label)
        ) * 0.5
        loss_D.backward()
        opt_D.step()

        # Train Generator
        opt_G.zero_grad()
        fake_vols  = G(real_vols, ages_norm)
        loss_G_adv = criterion_gan(D(fake_vols, ages_norm), real_label)
        loss_L1    = criterion_l1(fake_vols, real_vols) * LAMBDA_L1
        pred_age   = age_regressor(fake_vols).squeeze()
        loss_age   = criterion_age(pred_age, ages_raw) * LAMBDA_AGE
        loss_G     = loss_G_adv + loss_L1 + loss_age
        loss_G.backward()
        opt_G.step()

        epoch_loss_G   += loss_G_adv.item()
        epoch_loss_D   += loss_D.item()
        epoch_loss_L1  += loss_L1.item()
        epoch_loss_age += loss_age.item()

    scheduler_G.step()
    scheduler_D.step()

    n = len(dataloader)
    avg_loss_G   = epoch_loss_G   / n
    avg_loss_D   = epoch_loss_D   / n
    avg_loss_L1  = epoch_loss_L1  / n
    avg_loss_age = epoch_loss_age / n

    history['loss_G'].append(avg_loss_G)
    history['loss_D'].append(avg_loss_D)
    history['loss_L1'].append(avg_loss_L1)
    history['loss_age'].append(avg_loss_age)

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'loss_G={avg_loss_G:.4f} | '
          f'loss_D={avg_loss_D:.4f} | '
          f'loss_L1={avg_loss_L1:.4f} | '
          f'loss_age={avg_loss_age:.4f}')

    # Lưu best model
    if avg_loss_G < best_loss_G:
        best_loss_G = avg_loss_G
        torch.save({
            'epoch'       : epoch,
            'G_state'     : G.state_dict(),
            'D_state'     : D.state_dict(),
            'opt_G'       : opt_G.state_dict(),
            'opt_D'       : opt_D.state_dict(),
            'history'     : history,
            'age_min'     : dataset.age_min,
            'age_max'     : dataset.age_max,
            'best_loss_G' : best_loss_G,
            'volume_size' : VOLUME_SIZE,
        }, f'{OUTPUT_DIR}/best_model.pth')
        print(f'  → Best model saved! (loss_G={best_loss_G:.4f})')

print(f'\n=== TRAINING HOÀN THÀNH ===')
print(f'Best loss_G  : {best_loss_G:.4f}')
print(f'Model lưu tại: {OUTPUT_DIR}/best_model.pth')

Bắt đầu training 100 epochs...



Epoch 1/100:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch   1/100 | loss_G=1.6607 | loss_D=0.4339 | loss_L1=61.2084 | loss_age=0.0462
  → Best model saved! (loss_G=1.6607)


Epoch 2/100:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch   2/100 | loss_G=2.0542 | loss_D=0.3737 | loss_L1=29.8387 | loss_age=0.0287


...

Epoch  98/100 | loss_G=0.9064 | loss_D=0.6352 | loss_L1=1.9098 | loss_age=0.0117


Epoch 99/100:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch  99/100 | loss_G=0.8950 | loss_D=0.5885 | loss_L1=1.8835 | loss_age=0.0115


Epoch 100/100:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 100/100 | loss_G=0.9422 | loss_D=0.5830 | loss_L1=1.8836 | loss_age=0.0115

=== TRAINING HOÀN THÀNH ===
Best loss_G  : 0.7184
Model lưu tại: /kaggle/working/gan3d_normalized/best_model.pth
